<a href="https://colab.research.google.com/github/elig18/Study-case-Minastirith/blob/main/BILLETERIE_MINASTIRITH_gil.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Elisabeth Gil B3 IA

# CAS BILLETERIE – MINASTIRITH
## Manipulation avancée SQL / PL-SQL

---

**Projet :** La société MINASTIRITH souhaite informatiser sa billetterie (salles, billets, clients, factures).  
**Règles métier clés :**
- Un événement ne se déroule qu'à un emplacement à la fois
- Un billet = une seule place
- Vente interdite moins de **2h** avant l'événement
- Une facture peut couvrir 1 ou N billets

---

## Schéma relationnel (MCD → MLD)

```
CIVILITE      (id PK, libelle)
CLIENT        (id PK, nom, prenom, id_civilite FK→CIVILITE)
TELEPHONE     (id PK, numero, id_client FK→CLIENT)
SALLE         (id PK, nom, adresse, ville, codepostal, complement)
EVENEMENT     (id PK, libelle, description, id_salle FK→SALLE, nbplacesdisponibles, datebevenement)
CATEGORIE_PRIX(id PK, montant)
PLACE         (id PK, numero, gradin, rangee, id_salle FK→SALLE, id_catprix FK→CATEGORIE_PRIX)
BILLET        (id PK, code, id_client FK→CLIENT, id_place FK→PLACE, id_evenement FK→EVENEMENT)
FACTURE       (id PK, reference, date, id_client FK→CLIENT)
CONCERNER     (id_billet FK→BILLET, id_facture FK→FACTURE)  -- table d'asso N:N
```

---
# PARTIE 1 — SQL : Requêtes simples et complexes

## Question 1 — Script de création de la base de données

In [ ]:
%%sql
-- CREATION BASE BILLETERIE – MINASTIRITH

CREATE TABLE civilite (
    id         INT PRIMARY KEY,
    libelle    VARCHAR(50) NOT NULL
);

CREATE TABLE client (
    id          INT PRIMARY KEY,
    nom         VARCHAR(50)  NOT NULL,
    prenom      VARCHAR(50)  NOT NULL,
    id_civilite INT REFERENCES civilite(id)
);

CREATE TABLE telephone (
    id        INT PRIMARY KEY,
    numero    VARCHAR(60) NOT NULL,
    id_client INT NOT NULL REFERENCES client(id)
);

CREATE TABLE salle (
    id          INT PRIMARY KEY,
    nom         VARCHAR(50)  NOT NULL,
    adresse     VARCHAR(250),
    ville       VARCHAR(150),
    codepostal  CHAR(5),
    complement  VARCHAR(140)
);

CREATE TABLE evenement (
    id                   INT PRIMARY KEY,
    libelle              VARCHAR(60)  NOT NULL,
    description          TEXT,
    id_salle             INT NOT NULL REFERENCES salle(id),
    nbplacesdisponibles  INT DEFAULT 0,
    datebevenement       DATE NOT NULL
);

CREATE TABLE categorie_prix (
    id      INT PRIMARY KEY,
    montant FLOAT(10,2) NOT NULL
);

CREATE TABLE place (
    id         INT PRIMARY KEY,
    numero     INT  NOT NULL,
    gradin     CHAR(1),
    rangee     INT,
    id_salle   INT NOT NULL REFERENCES salle(id),
    id_catprix INT REFERENCES categorie_prix(id)
);

CREATE TABLE facture (
    id        INT PRIMARY KEY,
    reference VARCHAR(10) NOT NULL,
    date      DATE NOT NULL,
    id_client INT NOT NULL REFERENCES client(id)
);

CREATE TABLE billet (
    id           INT PRIMARY KEY,
    code         VARCHAR(250) NOT NULL,
    id_client    INT NOT NULL REFERENCES client(id),
    id_place     INT NOT NULL REFERENCES place(id),
    id_evenement INT REFERENCES evenement(id)
);

-- Table d'association billet <-> facture
-- (1 facture peut couvrir N billets, 1 billet peut figurer sur N factures)
CREATE TABLE concerner (
    id_billet  INT NOT NULL REFERENCES billet(id),
    id_facture INT NOT NULL REFERENCES facture(id),
    PRIMARY KEY (id_billet, id_facture)
);

## Question 2 — Nom, prénom et civilité de tous les clients (ordre alphabétique)

In [ ]:
%%sql
SELECT cl.nom       AS "Nom",
       cl.prenom    AS "Prénom",
       ci.libelle   AS "Civilité"
FROM   client   cl
JOIN   civilite ci ON ci.id = cl.id_civilite
ORDER BY cl.nom ASC, cl.prenom ASC;

## Question 3 — Nombre de téléphones par client
> Classé par nombre **décroissant**, puis nom/prénom **croissant**.

In [ ]:
%%sql
-- LEFT JOIN indispensable : conserve les clients sans téléphone (COUNT = 0)
SELECT cl.nom        AS "Nom",
       cl.prenom     AS "Prénom",
       COUNT(t.id)   AS "Nb téléphones"
FROM   client    cl
LEFT JOIN telephone t ON t.id_client = cl.id
GROUP BY cl.id, cl.nom, cl.prenom
ORDER BY COUNT(t.id) DESC,
         cl.nom       ASC,
         cl.prenom    ASC;

## Question 4 — Combien de clients n'ont pas de numéro de téléphone ?

In [ ]:
%%sql
-- Utilisation de NOT EXISTS : plus robuste que NOT IN avec des NULLs potentiels
SELECT COUNT(*) AS "Clients sans téléphone"
FROM   client cl
WHERE  NOT EXISTS (
    SELECT 1
    FROM   telephone t
    WHERE  t.id_client = cl.id
);

-- Alternative avec LEFT JOIN / IS NULL :
-- SELECT COUNT(*)
-- FROM   client cl
-- LEFT JOIN telephone t ON t.id_client = cl.id
-- WHERE  t.id IS NULL;

## Question 5 — Billets commandés mais non facturés, par client

In [ ]:
%%sql
SELECT cl.nom       AS "Nom",
       cl.prenom    AS "Prénom",
       COUNT(b.id)  AS "Billets non facturés"
FROM   client cl
JOIN   billet b  ON b.id_client = cl.id
WHERE  b.id NOT IN (
    SELECT c.id_billet FROM concerner c
)
GROUP BY cl.id, cl.nom, cl.prenom
ORDER BY cl.nom, cl.prenom;

## Question 6 — Événements et salle entre J+1 mois et J+1 an 1 mois (date décroissante)

In [ ]:
%%sql
-- Syntaxe Oracle (ADD_MONTHS / SYSDATE)
-- MySQL : DATE_ADD(NOW(), INTERVAL 1 MONTH) et DATE_ADD(NOW(), INTERVAL 13 MONTH)
SELECT e.libelle           AS "Événement",
       s.nom               AS "Salle",
       e.datebevenement    AS "Date"
FROM   evenement e
JOIN   salle     s ON s.id = e.id_salle
WHERE  e.datebevenement BETWEEN ADD_MONTHS(SYSDATE, 1)
                             AND ADD_MONTHS(SYSDATE, 13)
ORDER BY e.datebevenement DESC;

## Question 7 — Nombre de places achetées par client et par facture

In [ ]:
%%sql
SELECT cl.nom              AS "Nom",
       cl.prenom           AS "Prénom",
       f.reference         AS "Facture",
       f.date              AS "Date facture",
       COUNT(c.id_billet)  AS "Nb places"
FROM   client   cl
JOIN   facture  f  ON f.id_client  = cl.id
JOIN   concerner c ON c.id_facture = f.id
GROUP BY cl.id, cl.nom, cl.prenom, f.id, f.reference, f.date
ORDER BY cl.nom, cl.prenom, f.date;

## Question 8 — Facture de SACQUET Bilbon (4 places, « La chute du faucon noir »)

In [ ]:
%%sql
-- Étape 1 : vérification des IDs (à adapter selon les données réelles)
SELECT cl.id AS id_client,
       e.id  AS id_evenement,
       s.id  AS id_salle
FROM   client cl, evenement e, salle s
WHERE  cl.nom    = 'SACQUET'
  AND  cl.prenom = 'Bilbon'
  AND  e.libelle = 'La chute du faucon noir'
  AND  s.nom     = 'Le gouffre de Helm';

In [ ]:
%%sql
-- Étape 2 : création des 4 billets pour Bilbon
-- (hypothèse : id_client=1, id_evenement=10, places libres 101-104)
INSERT INTO billet (id, code, id_client, id_place, id_evenement)
    VALUES (101, 'BILLET-BILBON-1', 1, 101, 10);
INSERT INTO billet (id, code, id_client, id_place, id_evenement)
    VALUES (102, 'BILLET-BILBON-2', 1, 102, 10);
INSERT INTO billet (id, code, id_client, id_place, id_evenement)
    VALUES (103, 'BILLET-BILBON-3', 1, 103, 10);
INSERT INTO billet (id, code, id_client, id_place, id_evenement)
    VALUES (104, 'BILLET-BILBON-4', 1, 104, 10);

-- Étape 3 : création de la facture
INSERT INTO facture (id, reference, date, id_client)
    VALUES (1, 'FACT-0001', TO_DATE('27/10/2019','DD/MM/YYYY'), 1);

-- Étape 4 : rattachement billets -> facture
INSERT INTO concerner (id_billet, id_facture) VALUES (101, 1);
INSERT INTO concerner (id_billet, id_facture) VALUES (102, 1);
INSERT INTO concerner (id_billet, id_facture) VALUES (103, 1);
INSERT INTO concerner (id_billet, id_facture) VALUES (104, 1);

COMMIT;

## Question 9 — Gimli commande 8 places + facture

In [ ]:
%%sql
-- Utilisation d'une boucle PL/SQL pour insérer 8 billets + la facture en une passe
BEGIN
    -- 8 billets pour Gimli (id_client=2, places 105 à 112, même événement id=10)
    FOR i IN 1..8 LOOP
        INSERT INTO billet (id, code, id_client, id_place, id_evenement)
        VALUES (104 + i,
                'BILLET-GIMLI-' || i,
                2,
                104 + i,
                10);
    END LOOP;

    -- Facture de Gimli
    INSERT INTO facture (id, reference, date, id_client)
    VALUES (2, 'FACT-0002', TO_DATE('27/10/2019','DD/MM/YYYY'), 2);

    -- Association billets / facture
    FOR i IN 1..8 LOOP
        INSERT INTO concerner (id_billet, id_facture)
        VALUES (104 + i, 2);
    END LOOP;

    COMMIT;
END;
/

---
# PARTIE 2 — PL/SQL : Procédures, Fonctions et Triggers

## Question 1 — Procédure de génération de facture

**Paramètres :** `p_id_client`, `p_id_salle`, `p_id_evenement`  
**Traitement :** vérifie l'existence des 3 entités, crée la facture, rattache tous les billets non encore facturés.

In [ ]:
%%sql
CREATE OR REPLACE PROCEDURE generer_facture(
    p_id_client    IN INT,
    p_id_salle     IN INT,
    p_id_evenement IN INT
)
IS
    v_count_client INT;
    v_count_salle  INT;
    v_count_event  INT;
    v_id_facture   INT;
    v_ref_facture  VARCHAR(20);
BEGIN
    -- Vérifications d'existence
    SELECT COUNT(*) INTO v_count_client FROM client    WHERE id = p_id_client;
    SELECT COUNT(*) INTO v_count_salle  FROM salle     WHERE id = p_id_salle;
    SELECT COUNT(*) INTO v_count_event  FROM evenement WHERE id = p_id_evenement;

    IF v_count_client = 0 THEN
        RAISE_APPLICATION_ERROR(-20001,
            'Client introuvable (id=' || p_id_client || ')');
    END IF;
    IF v_count_salle = 0 THEN
        RAISE_APPLICATION_ERROR(-20002,
            'Salle introuvable (id=' || p_id_salle || ')');
    END IF;
    IF v_count_event = 0 THEN
        RAISE_APPLICATION_ERROR(-20003,
            'Événement introuvable (id=' || p_id_evenement || ')');
    END IF;

    -- Génération de l'ID et de la référence
    SELECT NVL(MAX(id), 0) + 1 INTO v_id_facture FROM facture;
    v_ref_facture := 'FACT-' || LPAD(v_id_facture, 5, '0');

    -- Création de la facture
    INSERT INTO facture (id, reference, date, id_client)
    VALUES (v_id_facture, v_ref_facture, SYSDATE, p_id_client);

    -- Rattacher les billets non encore facturés du client pour cet événement/salle
    INSERT INTO concerner (id_billet, id_facture)
    SELECT b.id, v_id_facture
    FROM   billet b
    JOIN   place  p ON p.id = b.id_place
    WHERE  b.id_client    = p_id_client
      AND  b.id_evenement = p_id_evenement
      AND  p.id_salle     = p_id_salle
      AND  b.id NOT IN (SELECT id_billet FROM concerner);

    COMMIT;
    DBMS_OUTPUT.PUT_LINE('Facture ' || v_ref_facture || ' créée avec succès.');

EXCEPTION
    WHEN OTHERS THEN
        ROLLBACK;
        DBMS_OUTPUT.PUT_LINE('Erreur : ' || SQLERRM);
        RAISE;
END generer_facture;
/

-- Exemple d'appel :
-- EXEC generer_facture(1, 5, 10);

## Question 2 — Procédure d'enregistrement d'une réservation

**Paramètres :** `p_id_client`, `p_id_evenement`, `p_nb_places`  
**Contrôles :** délai 2h, places suffisantes, sélection des places libres par numéro croissant.

In [ ]:
%%sql
CREATE OR REPLACE PROCEDURE reserver_places(
    p_id_client    IN INT,
    p_id_evenement IN INT,
    p_nb_places    IN INT
)
IS
    v_dispo     INT;
    v_date_evt  DATE;
    v_id_billet INT;
    v_id_place  INT;
    v_count     INT := 0;

    -- Curseur : places libres pour l'événement, triées par numéro croissant
    CURSOR c_places IS
        SELECT p.id
        FROM   place p
        WHERE  p.id_salle = (SELECT id_salle FROM evenement WHERE id = p_id_evenement)
          AND  p.id NOT IN (
                   SELECT b.id_place
                   FROM   billet b
                   WHERE  b.id_evenement = p_id_evenement
               )
        ORDER BY p.numero
        FETCH FIRST p_nb_places ROWS ONLY;
BEGIN
    -- Récupération de la date et du nombre de places dispo
    SELECT datebevenement, nbplacesdisponibles
    INTO   v_date_evt, v_dispo
    FROM   evenement
    WHERE  id = p_id_evenement;

    -- Règle métier : interdiction de vendre moins de 2h avant l'événement
    IF SYSDATE >= v_date_evt - (2/24) THEN
        RAISE_APPLICATION_ERROR(-20010,
            'Vente impossible : moins de 2h avant l''événement.');
    END IF;

    -- Contrôle disponibilité
    IF v_dispo < p_nb_places THEN
        RAISE_APPLICATION_ERROR(-20011,
            'Places insuffisantes : ' || v_dispo || ' disponible(s) seulement.');
    END IF;

    -- Récupération du dernier ID billet pour auto-incrément
    SELECT NVL(MAX(id), 0) INTO v_id_billet FROM billet;

    -- Création des billets
    OPEN c_places;
    LOOP
        FETCH c_places INTO v_id_place;
        EXIT WHEN c_places%NOTFOUND OR v_count >= p_nb_places;
        v_id_billet := v_id_billet + 1;
        v_count     := v_count + 1;
        INSERT INTO billet (id, code, id_client, id_place, id_evenement)
        VALUES (v_id_billet,
                'BLT-' || p_id_evenement || '-' || v_id_billet,
                p_id_client,
                v_id_place,
                p_id_evenement);
    END LOOP;
    CLOSE c_places;

    COMMIT;
    DBMS_OUTPUT.PUT_LINE(v_count || ' billet(s) créé(s) pour le client ' || p_id_client);

EXCEPTION
    WHEN OTHERS THEN
        ROLLBACK;
        IF c_places%ISOPEN THEN CLOSE c_places; END IF;
        DBMS_OUTPUT.PUT_LINE('Erreur réservation : ' || SQLERRM);
        RAISE;
END reserver_places;
/

-- Exemple d'appel :
-- EXEC reserver_places(2, 10, 8);

## Question 3 — Fonction : trouver les places contiguës disponibles

**Paramètres :** `p_id_evenement`, `p_gradin`, `p_rangee`, `p_nb_places`  
**Retour :** liste des IDs de places séparés par des virgules, ou `'0'` si pas assez de contiguës.

**Exemple :** Gradin B, Rangée 3 → places libres 32,33,34  
- Demande 4 → retourne `'0'`  
- Demande 2 → retourne `'32,33'`

In [ ]:
%%sql
CREATE OR REPLACE FUNCTION trouver_places_contiguës(
    p_id_evenement IN INT,
    p_gradin       IN CHAR,
    p_rangee       IN INT,
    p_nb_places    IN INT
) RETURN VARCHAR2
IS
    v_resultat  VARCHAR2(4000) := '';
    v_seq_len   INT := 0;
    v_found     BOOLEAN := FALSE;

    -- Places libres du gradin/rangée demandés, triées par numéro
    CURSOR c_places IS
        SELECT p.id, p.numero
        FROM   place p
        WHERE  p.id_salle = (SELECT id_salle FROM evenement WHERE id = p_id_evenement)
          AND  p.gradin   = p_gradin
          AND  p.rangee   = p_rangee
          AND  p.id NOT IN (
                   SELECT b.id_place FROM billet b
                   WHERE  b.id_evenement = p_id_evenement
               )
        ORDER BY p.numero;

    TYPE t_tab IS TABLE OF c_places%ROWTYPE;
    v_tab       t_tab;
    v_i         INT;
    v_seq_start INT;
BEGIN
    -- Chargement en mémoire (BULK COLLECT)
    OPEN c_places;
    FETCH c_places BULK COLLECT INTO v_tab;
    CLOSE c_places;

    -- Recherche de la première séquence contiguë de longueur >= p_nb_places
    v_i := 1;
    WHILE v_i <= v_tab.COUNT LOOP
        v_seq_start := v_i;
        v_seq_len   := 1;
        -- Avancer tant que les numéros sont consécutifs
        WHILE v_i < v_tab.COUNT
          AND v_tab(v_i + 1).numero = v_tab(v_i).numero + 1 LOOP
            v_i       := v_i + 1;
            v_seq_len := v_seq_len + 1;
        END LOOP;

        IF v_seq_len >= p_nb_places THEN
            -- On prend les p_nb_places premiers IDs (numéro le plus petit en premier)
            FOR j IN v_seq_start .. v_seq_start + p_nb_places - 1 LOOP
                v_resultat := v_resultat ||
                    CASE WHEN v_resultat IS NULL THEN '' ELSE ',' END ||
                    v_tab(j).id;
            END LOOP;
            v_found := TRUE;
            EXIT;  -- On prend la première séquence suffisante
        END IF;
        v_i := v_i + 1;
    END LOOP;

    IF NOT v_found THEN
        RETURN '0';
    END IF;
    RETURN v_resultat;
END trouver_places_contiguës;
/

-- Test :
SELECT trouver_places_contiguës(10, 'B', 3, 2) AS "Places disponibles" FROM DUAL;
-- => '32,33'

SELECT trouver_places_contiguës(10, 'B', 3, 4) AS "Places disponibles" FROM DUAL;
-- => '0' (pas assez de contiguës)

## Question 4 — Trigger : mise à jour du nombre de places disponibles

**Déclenchement :** `AFTER INSERT OR DELETE` sur la table `BILLET`  
- `INSERT` → décrémente `nbplacesdisponibles`  
- `DELETE` → incrémente `nbplacesdisponibles`

In [ ]:
%%sql
CREATE OR REPLACE TRIGGER trg_maj_places_dispo
AFTER INSERT OR DELETE ON billet
FOR EACH ROW   -- se déclenche une fois par ligne insérée/supprimée
BEGIN
    IF INSERTING THEN
        -- Nouveau billet : une place de moins disponible
        UPDATE evenement
        SET    nbplacesdisponibles = nbplacesdisponibles - 1
        WHERE  id = :NEW.id_evenement;
    END IF;

    IF DELETING THEN
        -- Billet supprimé : une place de plus disponible
        UPDATE evenement
        SET    nbplacesdisponibles = nbplacesdisponibles + 1
        WHERE  id = :OLD.id_evenement;
    END IF;
END trg_maj_places_dispo;
/

-- Note : :NEW et :OLD sont des pseudo-enregistrements PL/SQL
-- :NEW est disponible en INSERT, :OLD en DELETE, les deux en UPDATE

## Question 5 — Procédure rapport annuel (5 dernières années) avec curseurs

**Sortie attendue (DBMS_OUTPUT) :**
```
==============================
Année 2020
==============================
  Salle      : Le gouffre de Helm
  Événement  : La chute du faucon noir  (27/10/2020)
  Places vendues   : 120
  Places restantes : 30
  CA Réalisé       : 3 600,00 €
  ---
...
```

Utilise **deux curseurs imbriqués** comme demandé en Annexe B :
- Curseur simple sur les années
- Curseur paramétré sur les événements de chaque année

In [ ]:
%%sql
CREATE OR REPLACE PROCEDURE rapport_ventes_annuel
IS
    v_annee_courante INT;
    v_ca             FLOAT;


    -- Curseur 1 (SIMPLE) : années des 5 dernières

    CURSOR c_annees IS
        SELECT DISTINCT EXTRACT(YEAR FROM e.datebevenement) AS annee
        FROM   evenement e
        WHERE  EXTRACT(YEAR FROM e.datebevenement)
                   BETWEEN EXTRACT(YEAR FROM SYSDATE) - 4
                       AND EXTRACT(YEAR FROM SYSDATE)
        ORDER BY annee;


    -- Curseur 2 (PARAMÉTRÉ) : salles + événements pour une année donnée

    CURSOR c_evenements (p_annee INT) IS
        SELECT s.nom                AS nom_salle,
               e.libelle            AS nom_event,
               e.datebevenement     AS date_event,
               e.id                 AS id_event,
               e.nbplacesdisponibles AS restantes,
               (SELECT COUNT(*)
                FROM   billet b
                WHERE  b.id_evenement = e.id) AS vendues
        FROM   evenement e
        JOIN   salle     s ON s.id = e.id_salle
        WHERE  EXTRACT(YEAR FROM e.datebevenement) = p_annee
        ORDER BY s.nom, e.datebevenement;

    v_evt c_evenements%ROWTYPE;
BEGIN
    -- Boucle sur les années
    OPEN c_annees;
    LOOP
        FETCH c_annees INTO v_annee_courante;
        EXIT WHEN c_annees%NOTFOUND;

        DBMS_OUTPUT.PUT_LINE('');
        DBMS_OUTPUT.PUT_LINE('==============================');
        DBMS_OUTPUT.PUT_LINE('Année ' || v_annee_courante);
        DBMS_OUTPUT.PUT_LINE('==============================');

        -- Boucle sur les événements de l'année
        OPEN c_evenements(v_annee_courante);
        LOOP
            FETCH c_evenements INTO v_evt;
            EXIT WHEN c_evenements%NOTFOUND;

            -- Calcul du CA : somme des tarifs des billets vendus pour cet événement
            SELECT NVL(SUM(cp.montant), 0)
            INTO   v_ca
            FROM   billet        b
            JOIN   place         p  ON p.id    = b.id_place
            JOIN   categorie_prix cp ON cp.id  = p.id_catprix
            WHERE  b.id_evenement = v_evt.id_event;

            DBMS_OUTPUT.PUT_LINE('  Salle      : ' || v_evt.nom_salle);
            DBMS_OUTPUT.PUT_LINE('  Événement  : ' || v_evt.nom_event ||
                                 '  (' || TO_CHAR(v_evt.date_event,'DD/MM/YYYY') || ')');
            DBMS_OUTPUT.PUT_LINE('  Places vendues   : ' || v_evt.vendues);
            DBMS_OUTPUT.PUT_LINE('  Places restantes : ' || v_evt.restantes);
            DBMS_OUTPUT.PUT_LINE('  CA Réalisé       : ' ||
                                 TO_CHAR(v_ca,'FM999G990D00') || ' €');
            DBMS_OUTPUT.PUT_LINE('  ---');
        END LOOP;
        CLOSE c_evenements;
    END LOOP;
    CLOSE c_annees;

EXCEPTION
    WHEN OTHERS THEN
        IF c_annees%ISOPEN     THEN CLOSE c_annees;     END IF;
        IF c_evenements%ISOPEN THEN CLOSE c_evenements; END IF;
        DBMS_OUTPUT.PUT_LINE('Erreur rapport : ' || SQLERRM);
        RAISE;
END rapport_ventes_annuel;
/

-- Exécution :
SET SERVEROUTPUT ON;
EXEC rapport_ventes_annuel;

---
## Récapitulatif des points clés

| Concept | Utilisation dans ce cas |
|---|---|
| `LEFT JOIN` | Q3, Q4 : clients sans téléphone |
| `NOT EXISTS` / `NOT IN` | Q4, Q5 : billets non facturés |
| `ADD_MONTHS(SYSDATE, n)` | Q6 : fenêtre de dates |
| `RAISE_APPLICATION_ERROR` | P2-Q1, Q2 : erreurs métier |
| `SYSDATE >= date_evt - (2/24)` | P2-Q2 : règle des 2h |
| `BULK COLLECT` | P2-Q3 : chargement en mémoire pour algo contigu |
| `:NEW` / `:OLD` | P2-Q4 : trigger INSERT/DELETE |
| Curseur paramétré | P2-Q5 : `CURSOR c_evt(p_annee INT) IS ...` |
| `OPEN / FETCH / EXIT WHEN %NOTFOUND / CLOSE` | P2-Q2, Q5 : manipulation multi-lignes |
| `ROLLBACK` dans `EXCEPTION` | Toutes procédures : atomicité des transactions |